In [ ]:
# ==========================================
# IMPORTACIÓN DE LIBRERÍAS
# ==========================================

import pulp
import pandas as pd
import matplotlib.pyplot as plt

print("Librerías importadas correctamente.")

# Taller 1 - Programación Lineal

## Métodos Cuantitativos

**Problema:** Planeación de producción de la empresa JCR

**Objetivo:** Formular y resolver un modelo de optimización que determine el plan de compra de materias primas, producción, inventario y entregas durante cuatro periodos, maximizando la utilidad total.

**Herramienta de optimización:** PuLP  
**Lenguaje:** Python  
**Entorno:** Jupyter Notebook

## 1. Descripción del problema

La empresa de manufactura JCR produce tres tipos de productos: P1, P2 y P3.

La planeación de producción se realiza para cuatro periodos (meses). Durante estos periodos, la empresa debe determinar cuánto comprar de cada materia prima, cuánto producir de cada producto y cuánto almacenar, teniendo en cuenta:

- La evolución del precio de venta de los productos.
- La capacidad disponible de producción.
- Los requerimientos de materias primas.
- Los límites mensuales de compra.
- Los inventarios iniciales.
- La demanda de cada producto en cada periodo.
- Los costos de compra y almacenamiento.
- La posibilidad de realizar entregas atrasadas.
- Los tamaños mínimos de lote de producción.
- El inventario mínimo requerido al finalizar el cuarto periodo.

El objetivo es maximizar la utilidad total obtenida durante los cuatro periodos.

## 2. Parámetros del equipo

Algunos parámetros del problema dependen de la composición del equipo.

Sea:

- $s$: número de estudiantes del equipo.
- $d$: suma de las dos últimas cifras de las cédulas de los integrantes.

La tasa de incremento de los precios por periodo está dada por:

$$
\alpha = 0.01 + 0.005s
$$

La capacidad disponible de producción en cada periodo está dada por:

$$
C_t = 180 + (-1)^{t-1}\frac{d}{100}
$$

Estos valores se utilizarán para calcular los parámetros específicos del modelo.

In [ ]:
# Parámetros del equipo

s = 2
d = 92

# Tasa de incremento de precios por periodo
alpha = 0.01 + 0.005 * s

print(f"Número de estudiantes (s): {s}")
print(f"Parámetro d: {d}")
print(f"Tasa de incremento (alpha): {alpha:.2%}")

## 3. Productos

La empresa produce tres productos:

| Producto | Precio base | Horas de procesamiento por unidad |
|---|---:|---:|
| P1 | $600.000 | 3 |
| P2 | $550.000 | 4 |
| P3 | $700.000 | 2 |

El precio de cada producto en el periodo $t$ se calcula mediante:

$$
Precio_{p,t}=PrecioBase_p(1+\alpha)^{t-1}
$$

donde $t \in \{1,2,3,4\}$.

In [ ]:
# Conjunto de productos
productos = ["P1", "P2", "P3"]

# Precio base de cada producto en el periodo 1
precio_base = {
    "P1": 600_000,
    "P2": 550_000,
    "P3": 700_000
}

# Horas de procesamiento requeridas por unidad
horas_producto = {
    "P1": 3,
    "P2": 4,
    "P3": 2
}

### Periodos de planeación

La planeación se realiza durante cuatro periodos:

$$
T=\{1,2,3,4\}
$$

In [ ]:
# Periodos de planeación
periodos = [1, 2, 3, 4]

### Evolución de los precios

El precio de cada producto aumenta en cada periodo según la tasa $\alpha$:

$$
Precio_{p,t}=PrecioBase_p(1+\alpha)^{t-1}
$$

Por lo tanto, el precio del periodo 1 corresponde al precio base, mientras que los periodos siguientes incorporan el incremento acumulado.

In [ ]:
# Cálculo de precios por producto y periodo

precios = {}

for producto in productos:
    precios[producto] = {}

    for t in periodos:
        precios[producto][t] = (
            precio_base[producto] * (1 + alpha) ** (t - 1)
            )

precios

## 4. Capacidad de producción

La capacidad disponible de producción en cada periodo está dada por:

$$
C_t = 180 + (-1)^{t-1}\frac{d}{100}
$$

Por lo tanto:

$$
C_1 = 180+\frac{d}{100}
$$

$$
C_2 = 180-\frac{d}{100}
$$

$$
C_3 = 180+\frac{d}{100}
$$

$$
C_4 = 180-\frac{d}{100}
$$

Cada unidad producida requiere:

- P1: 3 horas.
- P2: 4 horas.
- P3: 2 horas.

In [ ]:
# Capacidad disponible de producción
# El enunciado expresa la capacidad en horas por semana,
# mientras que los periodos de planeación corresponden a meses.
# Se utiliza una conversión aproximada de 4.33 semanas por mes.

semanas_por_mes = 4.33

capacidad_semanal = {
    t: 180 + ((-1) ** (t - 1)) * d / 100
    for t in periodos
}

capacidad = {
    t: capacidad_semanal[t] * semanas_por_mes
    for t in periodos
}

print("Capacidad mensual:")
for t in periodos:
    print(f"Periodo {t}: {capacidad[t]:.2f} horas")

## 5. Materias primas

La producción utiliza dos materias primas:

- M1
- M2

El consumo de materias primas por unidad de producto es:

| Producto | M1 | M2 |
|---|---:|---:|
| P1 | 2 | 1 |
| P2 | 1 | 3 |
| P3 | 2 | 2 |

La disponibilidad máxima mensual de compra depende de $s$:

$$
M1_t \leq 600-12s
$$

$$
M2_t \leq 480+8s
$$

El inventario inicial es:

$$
M1_0=40
$$

$$
M2_0=30
$$

Los costos de compra son:

$$
c_{M1}=50.000
$$

$$
c_{M2}=70.000
$$

In [ ]:
# Materias primas
materias_primas = ["M1", "M2"]

# Consumo de materia prima por unidad de producto
consumo_mp = {
    "P1": {"M1": 2, "M2": 1},
    "P2": {"M1": 1, "M2": 3},
    "P3": {"M1": 2, "M2": 2}
}

# Inventario inicial de materias primas
inventario_inicial_mp = {
    "M1": 40,
    "M2": 30
}

# Costos de compra
costo_compra = {
    "M1": 50_000,
    "M2": 70_000
}

In [ ]:
# Disponibilidad máxima mensual de compra

max_compra = {
        "M1": 600 - 12 * s,
        "M2": 480 + 8 * s
    }

max_compra

## 6. Demanda

La demanda de cada producto para los cuatro periodos es:

| Periodo | P1 | P2 | P3 |
|---|---:|---:|---:|
| 1 | 120 | 60 | 72 |
| 2 | 72 | 80 | 60 |
| 3 | 100 | 130 | 80 |
| 4 | 60 | 62 | 68 |

El inventario inicial de productos terminados es:

| Producto | Inventario inicial |
|---|---:|
| P1 | 20 |
| P2 | 24 |
| P3 | 16 |

In [ ]:
# Demanda por producto y periodo
demanda = {
    "P1": {1: 120, 2: 72, 3: 100, 4: 60},
    "P2": {1: 60, 2: 80, 3: 130, 4: 62},
    "P3": {1: 72, 2: 60, 3: 80, 4: 68}
}

# Inventario inicial de productos terminados
inventario_inicial_producto = {
    "P1": 20,
    "P2": 24,
    "P3": 16
}

## 7. Costos de almacenamiento

El costo de almacenamiento por periodo corresponde al $s\%$ del precio base, tanto para los productos terminados como para las materias primas.

Por lo tanto, los costos de almacenamiento dependerán del parámetro $s$ y de los valores base establecidos en el problema.

## 8. Tamaños de lote

La producción debe realizarse en lotes determinados:

| Producto | Tamaño de lote |
|---|---:|
| P1 | 5 unidades |
| P2 | 1 unidad |
| P3 | 7 unidades |

Esto implica que las cantidades producidas deben respetar dichos tamaños de lote.

Por ejemplo:

$$
x_{P1,t} \in \{0,5,10,15,\ldots\}
$$

$$
x_{P3,t} \in \{0,7,14,21,\ldots\}
$$

In [ ]:
# Tamaño de lote por producto
tamano_lote = {
    "P1": 5,
    "P2": 1,
    "P3": 7
}

## 9. Inventario final y entregas atrasadas

Al finalizar el cuarto periodo debe existir un inventario mínimo de 20 unidades de cada producto:

$$
I_{P1,4} \geq 20
$$

$$
I_{P2,4} \geq 20
$$

$$
I_{P3,4} \geq 20
$$

Cuando no sea posible satisfacer la demanda de un periodo, el taller permite entregar los productos en el periodo siguiente.

En estos casos, el precio de venta se reduce en:

$$
0.1d\%
$$

Las entregas atrasadas deberán incorporarse al modelo como parte de las decisiones de optimización.

## 10. Objetivo de optimización

La empresa busca determinar un plan de:

- compra de materias primas,
- producción,
- almacenamiento,
- atención de la demanda,
- y entregas atrasadas,

que maximice la utilidad total durante los cuatro periodos.

De forma general:

$$
\boxed{
\text{Maximizar Utilidad}
=
\text{Ingresos}
-
\text{Costos de compra}
-
\text{Costos de almacenamiento}
-
\text{Costos asociados a entregas atrasadas}
}
$$

El siguiente paso será definir formalmente las variables de decisión y construir la función objetivo.

## 11. Variables de decisión

El modelo utiliza variables de decisión para representar las compras de materias primas,
la producción, los inventarios y las entregas atrasadas.

Las variables son:

- $Q_{m,t}$: cantidad de materia prima $m$ comprada en el periodo $t$.
- $X_{p,t}$: cantidad del producto $p$ producida en el periodo $t$.
- $Y_{p,t}$: número de lotes del producto $p$ producidos en el periodo $t$.
- $I^M_{m,t}$: inventario de materia prima $m$ al final del periodo $t$.
- $I^P_{p,t}$: inventario del producto $p$ al final del periodo $t$.
- $A_{p,t}$: cantidad de demanda del periodo $t$ que se entrega en el periodo $t+1$.

In [ ]:
# ==========================================
# VARIABLES DE DECISIÓN
# ==========================================

# Compra de materias primas
compra = pulp.LpVariable.dicts(
    "Compra",
    ((m, t) for m in materias_primas for t in periodos),
    lowBound=0,
    cat="Continuous"
)

# Producción de productos
produccion = pulp.LpVariable.dicts(
    "Produccion",
    ((p, t) for p in productos for t in periodos),
    lowBound=0,
    cat="Integer"
)

# Número de lotes producidos
lotes = pulp.LpVariable.dicts(
    "Lotes",
    ((p, t) for p in productos for t in periodos),
    lowBound=0,
    cat="Integer"
)

# Inventario de materias primas
inventario_mp = pulp.LpVariable.dicts(
    "InventarioMP",
    ((m, t) for m in materias_primas for t in periodos),
    lowBound=0,
    cat="Continuous"
)

# Inventario de productos terminados
inventario_producto = pulp.LpVariable.dicts(
    "InventarioProducto",
    ((p, t) for p in productos for t in periodos),
    lowBound=0,
    cat="Integer"
)

# Demanda atrasada
# Solo puede existir atraso en los periodos 1, 2 y 3,
# porque el horizonte termina en el periodo 4.
atraso = pulp.LpVariable.dicts(
    "Atraso",
    ((p, t) for p in productos for t in periodos[:-1]),
    lowBound=0,
    cat="Integer"
)

print("Variables de decisión creadas correctamente.")

In [ ]:
# ==========================================
# CREACIÓN DEL MODELO
# ==========================================

modelo = pulp.LpProblem(
    "Planeacion_Produccion_JCR",
    pulp.LpMaximize
)

print("Modelo de optimización creado correctamente.")

In [ ]:
# ==========================================
# RESTRICCIONES DE TAMAÑO DE LOTE
# ==========================================

for p in productos:
    for t in periodos:
        modelo += (
            produccion[p, t] == tamano_lote[p] * lotes[p, t],
            f"Lote_{p}_{t}"
        )

### 11.1. Compra de materias primas

Sea:

$$
Q_{m,t} = \text{cantidad de materia prima }m\text{ comprada en el periodo }t
$$

donde:

$$
m \in \{M1,M2\}
$$

y:

$$
t \in \{1,2,3,4\}
$$

Estas variables representan las decisiones mensuales de compra de materias primas.

### 11.2. Producción

Sea:

$$
X_{p,t} = \text{cantidad del producto }p\text{ producida en el periodo }t
$$

donde:

$$
p \in \{P1,P2,P3\}
$$

y:

$$
t \in \{1,2,3,4\}
$$

Estas variables estarán sujetas a las restricciones de capacidad y tamaño de lote.

### 11.3. Inventario de materias primas

Sea:

$$
I^M_{m,t}
=
\text{inventario de la materia prima }m
\text{ al final del periodo }t
$$

Este inventario permite trasladar materias primas no utilizadas de un periodo al siguiente.

### 11.4. Inventario de productos terminados

Sea:

$$
I^P_{p,t}
=
\text{inventario del producto }p
\text{ al final del periodo }t
$$

Este inventario permite utilizar productos producidos en un periodo para atender demanda en periodos posteriores.

### 11.5. Demanda entregada de forma atrasada

Sea:

$$
A_{p,t}
=
\text{cantidad del producto }p
\text{ correspondiente a la demanda del periodo }t
\text{ que se entrega en el periodo }t+1
$$

Estas unidades generan un ingreso reducido debido al descuento establecido en el problema.

El descuento es:

$$
0.1d\%
$$

Para nuestro equipo:

$$
0.1(92)\%=9.2\%
$$

Por lo tanto, una unidad entregada de forma atrasada genera el:

$$
1-0.092=0.908
$$

del precio normal.

### 11.6. Número de lotes producidos

Para representar los tamaños de lote mediante un modelo lineal, definimos:

$$
Y_{p,t}
=
\text{número de lotes del producto }p
\text{ producidos en el periodo }t
$$

La cantidad producida se relaciona con el número de lotes mediante:

$$
X_{p,t}=L_pY_{p,t}
$$

donde $L_p$ representa el tamaño de lote del producto.

Por tanto:

- $L_{P1}=5$
- $L_{P2}=1$
- $L_{P3}=7$

## 12. Función objetivo

El objetivo es maximizar la utilidad total durante los cuatro periodos.

La utilidad se define como:

$$
\text{Utilidad}
=
\text{Ingresos por ventas}
-
\text{Costos de compra}
-
\text{Costos de almacenamiento}
$$

### Ingresos por ventas normales

La cantidad de demanda atendida normalmente corresponde a:

$$
D_{p,t}-A_{p,t}
$$

Por tanto:

$$
Ingresos_{normal}
=
\sum_{p}\sum_t
P_{p,t}(D_{p,t}-A_{p,t})
$$

### Ingresos por entregas atrasadas

Las unidades entregadas con un periodo de atraso reciben un descuento del $0.1d\%$.

Para nuestro equipo:

$$
0.1(92)\%=9.2\%
$$

Por tanto:

$$
Ingresos_{atrasados}
=
\sum_p\sum_{t=1}^{3}
P_{p,t}(1-0.092)A_{p,t}
$$

### Costos de compra

$$
CostoCompra
=
\sum_m\sum_t c_mQ_{m,t}
$$

### Costos de almacenamiento

Para los productos terminados, el costo de almacenamiento corresponde al $s\%$ del precio base.

Para las materias primas, el enunciado no proporciona un precio base independiente. Se utilizarán los costos de compra como referencia para esta interpretación.

Finalmente:

$$
\boxed{
\max Z =
Ingresos_{normal}
+
Ingresos_{atrasados}
-
CostoCompra
-
CostoAlmacenamiento
}
$$

In [ ]:
# Costos de almacenamiento de productos terminados
costo_alm_producto = {
    "P1": s / 100 * precio_base["P1"],
    "P2": s / 100 * precio_base["P2"],
    "P3": s / 100 * precio_base["P3"]
}

# Para materias primas se utiliza el costo de compra
# como referencia para el precio base de almacenamiento.
costo_alm_mp = {
    "M1": s / 100 * costo_compra["M1"],
    "M2": s / 100 * costo_compra["M2"]
}

print("Costo almacenamiento productos:", costo_alm_producto)
print("Costo almacenamiento materias primas:", costo_alm_mp)

## 13. Restricción de capacidad

Cada producto requiere una cantidad determinada de horas de procesamiento.

La capacidad disponible en el periodo $t$ es $C_t$.

Por tanto:

$$
\sum_p h_pX_{p,t}\leq C_t
$$

donde $h_p$ representa las horas necesarias para producir una unidad del producto $p$.

Para cada periodo:

$$
3X_{P1,t}
+
4X_{P2,t}
+
2X_{P3,t}
\leq C_t
$$

In [ ]:
# Restricciones de capacidad de producción

for t in periodos:
    horas_utilizadas = sum(
        horas_producto[p] * produccion[p, t]
        for p in productos
    )

    modelo += (
        horas_utilizadas <= capacidad[t],
        f"Capacidad_{t}"
    )

## 14. Balance de materias primas

Para cada materia prima y periodo se debe conservar el balance:

$$
InventarioInicial
+
Compras
-
Consumo
=
InventarioFinal
$$

El consumo de cada materia prima depende de la producción de los productos.

Para cada materia prima $m$:

$$
I^M_{m,t}
=
I^M_{m,t-1}
+
Q_{m,t}
-
\sum_p a_{p,m}X_{p,t}
$$

donde $a_{p,m}$ representa la cantidad de materia prima $m$ necesaria para producir una unidad del producto $p$.

En el primer periodo se utilizan los inventarios iniciales proporcionados por el problema.

In [ ]:
# Balance de materias primas

for m in materias_primas:
    for t in periodos:

        if t == 1:
            inventario_anterior = inventario_inicial_mp[m]
        else:
            inventario_anterior = inventario_mp[m, t - 1]

        consumo = sum(
            consumo_mp[p][m] * produccion[p, t]
            for p in productos
        )

        modelo += (
            inventario_anterior
            + compra[m, t]
            - consumo
            == inventario_mp[m, t],
            f"BalanceMP_{m}_{t}"
        )

In [ ]:
# Límites máximos de compra de materias primas

for m in materias_primas:
    for t in periodos:
        modelo += (
            compra[m, t] <= max_compra[m],
            f"MaxCompra_{m}_{t}"
        )

## 15. Balance de productos terminados

El inventario de productos terminados se actualiza considerando:

- inventario disponible del periodo anterior;
- producción del periodo actual;
- demanda del periodo actual;
- demanda atrasada proveniente del periodo anterior;
- demanda actual que será entregada en el siguiente periodo.

Para los periodos intermedios:

$$
I^P_{p,t}
=
I^P_{p,t-1}
+
X_{p,t}
-
D_{p,t}
+
A_{p,t}
-
A_{p,t-1}
$$

donde:

- $I^P_{p,t}$ es el inventario final;
- $X_{p,t}$ es la producción;
- $D_{p,t}$ es la demanda;
- $A_{p,t}$ es la demanda del periodo $t$ entregada en $t+1$.

En el primer periodo:

$$
I^P_{p,1}
=
I^P_{p,0}
+
X_{p,1}
-
D_{p,1}
+
A_{p,1}
$$

En el cuarto periodo no se permite generar nuevos atrasos, ya que el horizonte de planeación termina allí:

$$
I^P_{p,4}
=
I^P_{p,3}
+
X_{p,4}
-
D_{p,4}
-
A_{p,3}
$$

In [ ]:
# Balance de productos terminados

for p in productos:
    for t in periodos:

        if t == 1:
            inventario_anterior = inventario_inicial_producto[p]

            modelo += (
                inventario_anterior
                + produccion[p, t]
                - demanda[p][t]
                + atraso[p, t]
                == inventario_producto[p, t],
                f"BalanceProducto_{p}_{t}"
            )

        elif t < 4:
            modelo += (
                inventario_producto[p, t - 1]
                + produccion[p, t]
                - demanda[p][t]
                + atraso[p, t]
                - atraso[p, t - 1]
                == inventario_producto[p, t],
                f"BalanceProducto_{p}_{t}"
            )

        else:
            modelo += (
                inventario_producto[p, t - 1]
                + produccion[p, t]
                - demanda[p][t]
                - atraso[p, t - 1]
                == inventario_producto[p, t],
                f"BalanceProducto_{p}_{t}"
            )

In [ ]:
# Una cantidad atrasada no puede superar la demanda del periodo correspondiente

for p in productos:
    for t in periodos[:-1]:
        modelo += (
            atraso[p, t] <= demanda[p][t],
            f"MaxAtraso_{p}_{t}"
        )

## 16. Inventario mínimo al finalizar el horizonte

Al finalizar el cuarto periodo, la empresa debe conservar como mínimo 20 unidades de cada producto:

$$
I^P_{p,4}\geq20
\qquad \forall p
$$

Esta restricción garantiza que el plan de producción no consuma completamente las existencias al finalizar el horizonte.

In [ ]:
# Inventario mínimo de productos al final del periodo 4

for p in productos:
    modelo += (
        inventario_producto[p, 4] >= 20,
        f"InventarioMinimo_{p}"
    )

In [ ]:
# Ingresos por ventas normales

ingresos_normales = sum(
    precios[p][t] * (
        demanda[p][t] - atraso[p, t]
        if t < 4
        else demanda[p][t]
    )
    for p in productos
    for t in periodos
)

In [ ]:
# Ingresos por entregas atrasadas
# El descuento para nuestro equipo es del 9.2%

descuento_atraso = 0.1 * d / 100

ingresos_atrasados = sum(
    precios[p][t] * (1 - descuento_atraso) * atraso[p, t]
    for p in productos
    for t in periodos[:-1]
)

In [ ]:
# Costos de compra de materias primas

costos_compra = sum(
    costo_compra[m] * compra[m, t]
    for m in materias_primas
    for t in periodos
)

In [ ]:
# Costos de almacenamiento de materias primas

costos_almacenamiento_mp = sum(
    costo_alm_mp[m] * inventario_mp[m, t]
    for m in materias_primas
    for t in periodos
)

In [ ]:
# Costos de almacenamiento de productos terminados

costos_almacenamiento_producto = sum(
    costo_alm_producto[p] * inventario_producto[p, t]
    for p in productos
    for t in periodos
)

In [ ]:
costos_almacenamiento = (
    costos_almacenamiento_mp
    + costos_almacenamiento_producto
)

In [ ]:
# Función objetivo: maximizar la utilidad total

utilidad = (
    ingresos_normales
    + ingresos_atrasados
    - costos_compra
    - costos_almacenamiento
)

modelo += utilidad

In [ ]:
print("Número de variables:", len(modelo.variables()))
print("Número de restricciones:", len(modelo.constraints))

In [ ]:
# ==========================================
# RESOLVER EL MODELO
# ==========================================

resultado = modelo.solve(pulp.PULP_CBC_CMD(msg=True))

print("Estado del modelo:", pulp.LpStatus[modelo.status])

In [ ]:
# ==========================================
# COMPROBACIONES DE LA SOLUCIÓN
# ==========================================

if pulp.LpStatus[modelo.status] != "Optimal":
    raise RuntimeError(
        f"El solver no encontró una solución óptima. Estado: {pulp.LpStatus[modelo.status]}"
    )

print("✓ El solver encontró una solución óptima.")

# Comprobar tamaños de lote
for p in productos:
    for t in periodos:
        produccion_val = pulp.value(produccion[p, t])
        lotes_val = pulp.value(lotes[p, t])
        esperado = tamano_lote[p] * lotes_val

        assert abs(produccion_val - esperado) <= 1e-6, (
            f"Error de lote en {p}, periodo {t}: "
            f"producción={produccion_val}, esperado={esperado}"
        )

print("✓ Tamaños de lote respetados.")


In [ ]:
print("==========================================")
print("DIAGNÓSTICO DEL SOLVER")
print("==========================================")

print("Código de estado:", modelo.status)
print("Estado:", pulp.LpStatus[modelo.status])
print("Resultado devuelto:", resultado)

In [ ]:
# ==========================================
# RESULTADO DE LA FUNCIÓN OBJETIVO
# ==========================================

print(f"Utilidad óptima: ${pulp.value(modelo.objective):,.2f}")

In [ ]:
# ==========================================
# PLAN DE PRODUCCIÓN
# ==========================================

resultado_produccion = pd.DataFrame(
    {
        t: [
            pulp.value(produccion[p, t])
            for p in productos
        ]
        for t in periodos
    },
    index=productos
)

resultado_produccion.columns.name = "Periodo"

print("Producción óptima:")
display(resultado_produccion)

In [ ]:
# ==========================================
# PLAN DE COMPRAS
# ==========================================

resultado_compras = pd.DataFrame(
    {
        t: [
            pulp.value(compra[m, t])
            for m in materias_primas
        ]
        for t in periodos
    },
    index=materias_primas
)

resultado_compras.columns.name = "Periodo"

print("Compras óptimas de materias primas:")
display(resultado_compras)

In [ ]:
# ==========================================
# INVENTARIO DE MATERIAS PRIMAS
# ==========================================

resultado_inventario_mp = pd.DataFrame(
    {
        t: [
            pulp.value(inventario_mp[m, t])
            for m in materias_primas
        ]
        for t in periodos
    },
    index=materias_primas
)

resultado_inventario_mp.columns.name = "Periodo"

print("Inventario de materias primas:")
display(resultado_inventario_mp)

In [ ]:
# ==========================================
# INVENTARIO DE PRODUCTOS TERMINADOS
# ==========================================

resultado_inventario_producto = pd.DataFrame(
    {
        t: [
            pulp.value(inventario_producto[p, t])
            for p in productos
        ]
        for t in periodos
    },
    index=productos
)

resultado_inventario_producto.columns.name = "Periodo"

print("Inventario de productos terminados:")
display(resultado_inventario_producto)

In [ ]:
# ==========================================
# DEMANDA ATRASADA
# ==========================================

resultado_atrasos = pd.DataFrame(
    {
        t: [
            pulp.value(atraso[p, t])
            for p in productos
        ]
        for t in periodos[:-1]
    },
    index=productos
)

resultado_atrasos.columns.name = "Demanda del periodo"

print("Demanda entregada con un periodo de atraso:")
display(resultado_atrasos)

In [ ]:
# Comprobar capacidad, compras e inventario final

for t in periodos:
    horas_usadas = sum(
        horas_producto[p] * pulp.value(produccion[p, t])
        for p in productos
    )

    assert horas_usadas <= capacidad[t] + 1e-6, (
        f"Se excedió la capacidad en el periodo {t}"
    )

    print(
        f"✓ Capacidad periodo {t}: "
        f"{horas_usadas:.2f} / {capacidad[t]:.2f} horas"
    )

for m in materias_primas:
    for t in periodos:
        compra_val = pulp.value(compra[m, t])

        assert compra_val <= max_compra[m] + 1e-6, (
            f"Se excedió la compra máxima de {m} en el periodo {t}"
        )

print("✓ Límites de compra respetados.")

for p in productos:
    inventario_final = pulp.value(inventario_producto[p, 4])

    assert inventario_final >= 20 - 1e-6, (
        f"Inventario final insuficiente para {p}"
    )

    print(
        f"✓ Inventario final {p}: "
        f"{inventario_final:.2f} unidades"
    )

print("✓ Todas las comprobaciones principales fueron superadas.")


In [ ]:
# ==========================================
# RESUMEN FINAL
# ==========================================

print("=" * 60)
print("RESUMEN DE LA OPTIMIZACIÓN")
print("=" * 60)
print(f"Estado: {pulp.LpStatus[modelo.status]}")
print(f"Utilidad óptima: ${pulp.value(modelo.objective):,.2f}")

print("\nInventario final:")
for p in productos:
    valor = pulp.value(inventario_producto[p, 4])
    print(f"  {p}: {valor:.2f} unidades")

print("\nDescuento por atraso:")
print(f"  {descuento_atraso:.2%}")


# Conclusiones

El modelo de programación lineal entera permitió determinar un plan óptimo
de producción, compra de materias primas e inventarios para los cuatro
periodos del horizonte de planeación.

El modelo considera:

- capacidad limitada de producción;
- requerimientos de materias primas;
- límites máximos de compra;
- tamaños mínimos de lote;
- inventarios iniciales;
- posibilidad de entregar demanda con un periodo de atraso;
- descuento del 9.2% sobre las ventas atrasadas;
- costos de compra y almacenamiento;
- inventario mínimo de 20 unidades de cada producto al finalizar el periodo 4.

La solución obtenida mediante PuLP y el solver CBC maximiza la utilidad
total bajo las restricciones definidas.

La solución debe considerarse válida únicamente si el solver reporta
estado `Optimal` y las comprobaciones de capacidad, compras, tamaños de
lote e inventario final son satisfechas.